In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
from sklearn.metrics import precision_score 



# Class

In [ ]:

class ResultAnalyzer:
    """
    A class to analyze and visualize prediction results from a CSV file.
    It calculates and plots location distributions, accuracy, precision, and sample counts.
    """
    def __init__(self, file_path, actual_location_col='true_label',
                 predicted_col_name='pred_1', correct_prediction_col='correct_prediction'):
        """
        Initializes the ResultAnalyzer.

        Args:
            file_path (str): The path to the CSV file containing prediction data.
            actual_location_col (str): The name of the column with true labels.
            predicted_col_name (str): The name of the column with predicted labels.
            correct_prediction_col (str): The name of the column indicating if the prediction was correct (boolean or 0/1).
        """
        self.file_path = file_path
        self.actual_location_col = actual_location_col
        self.predicted_col_name = predicted_col_name
        self.correct_prediction_col = correct_prediction_col
        self.df = None
        self.result_df_by_location = None
        self.overall_accuracy = None
        self.macro_precision = None
        self.weighted_precision = None


        self._load_data()
        self._calculate_metrics_by_location() # Calculate metrics upon initialization

    def _load_data(self):
        """Loads data from the CSV file."""
        try:
            self.df = pd.read_csv(self.file_path)
            print("Data loaded successfully. First 5 rows:")
            print(self.df.head())
            # Ensure the correct_prediction column exists if it's used for accuracy
            if self.correct_prediction_col not in self.df.columns:
                if self.actual_location_col in self.df.columns and self.predicted_col_name in self.df.columns:
                    self.df[self.correct_prediction_col] = (self.df[self.actual_location_col] == self.df[self.predicted_col_name]).astype(int)
                    print(f"\n'{self.correct_prediction_col}' column created.")
                else:
                    raise ValueError(f"'{self.correct_prediction_col}' column not found and cannot be created without "
                                     f"'{self.actual_location_col}' and '{self.predicted_col_name}' columns.")
        except FileNotFoundError:
            print(f"Error: File not found at {self.file_path}")
            raise
        except Exception as e:
            print(f"Error loading data: {e}")
            raise

    def _get_top_n_categories_for_pie(self, series, n=6):
        """
        Helper function to get top N categories from a pandas Series for pie charts,
        grouping the rest as "Other".
        """
        counts = series.value_counts()
        if len(counts) > n:
            top_n_series = counts.nlargest(n)
            other_count = counts.iloc[n:].sum()
            top_n_series = pd.concat([top_n_series, pd.Series({'Other': other_count})])
            return top_n_series
        return counts

    def _create_pie_chart(self, ax, data, title):
        """Helper function to create a single pie chart."""
        labels = [f"{idx} ({int(count)})" for idx, count in zip(data.index, data.values)]
        wedges, texts, autotexts = ax.pie(
            data,
            autopct='%1.1f%%',
            startangle=90,
            pctdistance=0.85,
            colors=plt.cm.tab10.colors[:len(data)]
        )
        ax.legend(
            wedges,
            labels,
            title="Location IDs (count)",
            loc="upper left",
            bbox_to_anchor=(1, 0, 0.5, 1)
        )
        ax.set_title(title)

    def plot_location_distribution(self, n_pie_categories=6):
        """
        Generates and displays two pie charts showing the distribution of
        actual and predicted locations (Top N + Other).
        """
        if self.df is None:
            print("Data not loaded. Call load_data() first.")
            return

        actual_top_n = self._get_top_n_categories_for_pie(self.df[self.actual_location_col], n=n_pie_categories)
        predicted_top_n = self._get_top_n_categories_for_pie(self.df[self.predicted_col_name], n=n_pie_categories)

        plt.figure(figsize=(16, 8))

        plt.subplot(1, 2, 1)
        self._create_pie_chart(plt.gca(), actual_top_n, f'Actual Location ID Distribution\n(Top {n_pie_categories} + Other)')

        plt.subplot(1, 2, 2)
        self._create_pie_chart(plt.gca(), predicted_top_n, f'Predicted Location ID Distribution\n(Top {n_pie_categories} + Other)')

        plt.tight_layout()
        plt.show()

    def _calculate_metrics_by_location(self):
        """
        Calculates accuracy, precision, and counts per location ID.
        Also calculates overall accuracy, and macro/weighted precision.
        """
        if self.df is None:
            print("Data not loaded.")
            return

        accuracy_by_location = self.df.groupby(self.actual_location_col)[self.correct_prediction_col].mean()
        counts_by_location = self.df.groupby(self.actual_location_col).size()

        unique_labels = sorted(self.df[self.actual_location_col].unique())


        # Calculate precision per class
        precision_by_location_array = precision_score(
            self.df[self.actual_location_col],
            self.df[self.predicted_col_name],
            labels=unique_labels,
            average=None,
            zero_division=0
        )
        precision_series = pd.Series(precision_by_location_array, index=unique_labels)


        self.result_df_by_location = pd.DataFrame({
            'accuracy': accuracy_by_location, 
            'count': counts_by_location
        })
        self.result_df_by_location['precision'] = precision_series
        self.result_df_by_location['precision'] = self.result_df_by_location['precision'].fillna(0)


        # Overall metrics
        self.overall_accuracy = self.df[self.correct_prediction_col].mean() * 100

        # Precision metrics
        self.macro_precision = self.result_df_by_location['precision'].mean() * 100
        if self.result_df_by_location['count'].sum() > 0: # Weight by actual counts for consistency
            self.weighted_precision = (self.result_df_by_location['precision'] * self.result_df_by_location['count']).sum() / \
                                      self.result_df_by_location['count'].sum() * 100
        else:
            self.weighted_precision = 0

        print("\nMetrics calculated by location.")


    def _get_top_n_with_others_for_bars(self, n=50, sort_by='count'):
        """
        Helper function to prepare data for the bar charts (Top N + Others for accuracy, precision, counts).
        """
        if self.result_df_by_location is None:
            self._calculate_metrics_by_location()
            if self.result_df_by_location is None:
                return None

        sorted_df = self.result_df_by_location.sort_values(sort_by, ascending=False)
        top_n = sorted_df.iloc[:n]
        others = sorted_df.iloc[n:]

        if not others.empty:
            others_total_count = others['count'].sum()
            others_weighted_accuracy = (others['accuracy'] * others['count']).sum() / others_total_count if others_total_count > 0 else 0
            others_weighted_precision = (others['precision'] * others['count']).sum() / others_total_count if others_total_count > 0 else 0


            others_row = pd.DataFrame({
                'accuracy': [others_weighted_accuracy],
                'count': [others_total_count],
                'precision': [others_weighted_precision]
            }, index=['Others'])
            return pd.concat([top_n, others_row])
        return top_n

    def plot_metrics_by_location(self, n_bar_categories=50, sort_bar_charts_by='count'):
        """
        Generates and displays bar charts for accuracy, precision, and sample counts per location ID.
        """
        if self.result_df_by_location is None:
            print("Metrics not calculated. Call _calculate_metrics_by_location() or ensure data is loaded.")
            return

        display_data = self._get_top_n_with_others_for_bars(n=n_bar_categories, sort_by=sort_bar_charts_by)
        if display_data is None:
            return

        plt.figure(figsize=(18, 24)) # Increased figure height for 4 plots

        # 1. Accuracy bar chart
        plt.subplot(3, 1, 1)
        bars_accuracy = plt.bar(
            display_data.index.astype(str),
            display_data['accuracy'] * 100,
            color=plt.cm.viridis(np.linspace(0, 1, len(display_data)))
        )
        for bar in bars_accuracy:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 1, f'{height:.1f}%',
                       ha='center', va='bottom', rotation=0, fontsize=8)
        plt.title(f'Prediction Accuracy by Location ID (Top {n_bar_categories} by {sort_bar_charts_by} + Others)', fontsize=16)
        #plt.xlabel('Location ID', fontsize=14) # X-label only for the last plot
        plt.ylabel('Accuracy (%)', fontsize=14)
        plt.ylim(0, 105)
        plt.xticks(rotation=45, ha="right", fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.7)


        # 2. Precision bar chart
        plt.subplot(3, 1, 2)
        bars_precision = plt.bar(
            display_data.index.astype(str),
            display_data['precision'] * 100,
            color=plt.cm.magma(np.linspace(0, 1, len(display_data))) # Different colormap
        )
        for bar in bars_precision:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 1, f'{height:.1f}%',
                       ha='center', va='bottom', rotation=0, fontsize=8)
        plt.title(f'Precision by Location ID (Top {n_bar_categories} by {sort_bar_charts_by} + Others)', fontsize=16)
        #plt.xlabel('Location ID', fontsize=14)
        plt.ylabel('Precision (%)', fontsize=14)
        plt.ylim(0, 105)
        plt.xticks(rotation=45, ha="right", fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.7)


        # 4. Sample count bar chart
        plt.subplot(3, 1, 3)
        bars_count = plt.bar(
            display_data.index.astype(str),
            display_data['count'],
            color=plt.cm.plasma(np.linspace(0, 1, len(display_data)))
        )
        for bar in bars_count:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + max(display_data['count'])*0.02,
                       f'{int(height)}', ha='center', va='bottom', fontsize=10)
        plt.title(f'Sample Count by Location ID (Top {n_bar_categories} by {sort_bar_charts_by} + Others)', fontsize=16)
        plt.xlabel('Location ID', fontsize=14) # X-label only for the last plot
        plt.ylabel('Sample Count', fontsize=14)
        plt.xticks(rotation=45, ha="right", fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.7)

        plt.figtext(0.5, 0.01,
                    f'Overall Acc: {self.overall_accuracy:.2f}% | '
                    f'Weighted Precision: {self.weighted_precision:.2f}%',
                    ha='center', fontsize=11, bbox=dict(facecolor='lightgray', alpha=0.5))

        plt.tight_layout(rect=[0, 0.04, 1, 0.97]) # Adjust rect to make space for figtext
        plt.show()

    def run_full_analysis(self, n_pie_categories=6, n_bar_categories=50, sort_bar_charts_by='count'):
        """
        Runs the full analysis pipeline: loads data, plots distributions, and plots metrics.
        """
        if self.df is None:
            print("Data could not be loaded. Aborting analysis.")
            return

        print("--- Location Distribution Analysis ---")
        self.plot_location_distribution(n_pie_categories=n_pie_categories)

        print("\n--- Metrics by Location Analysis ---")
        if self.result_df_by_location is None:
             self._calculate_metrics_by_location() # Should be already called in __init__
        if self.result_df_by_location is not None:
            print(f"\nOverall Prediction Accuracy: {self.overall_accuracy:.2f}%")
            print(f"Weighted Average Precision: {self.weighted_precision:.2f}%")
            self.plot_metrics_by_location(n_bar_categories=n_bar_categories, sort_bar_charts_by=sort_bar_charts_by)
        else:
            print("Metrics calculation failed. Skipping metrics plot.")

# LSTM

In [ ]:
file_path = '../../runs/dtepr_lstm_ce/dtepr_benchmark_test_predictions.csv' 
actual_col = 'true_label'
predicted_col = 'pred_1'

analyzer = ResultAnalyzer(file_path,
                          actual_location_col=actual_col,
                          predicted_col_name=predicted_col)

analyzer.run_full_analysis(n_pie_categories=5, n_bar_categories=15, sort_bar_charts_by='count')

# Mamba

In [ ]:
file_path = '../../runs/dtepr_mamba_ce/dtepr_benchmark_test_predictions.csv' 
actual_col = 'true_label'
predicted_col = 'pred_1'

analyzer = ResultAnalyzer(file_path,
                          actual_location_col=actual_col,
                          predicted_col_name=predicted_col)

analyzer.run_full_analysis(n_pie_categories=5, n_bar_categories=15, sort_bar_charts_by='count')

# MHSA

In [ ]:
file_path = '../../runs/dtepr_mhsa_ce/dtepr_benchmark_test_predictions.csv' 
actual_col = 'true_label'
predicted_col = 'pred_1'

analyzer = ResultAnalyzer(file_path,
                          actual_location_col=actual_col,
                          predicted_col_name=predicted_col)

analyzer.run_full_analysis(n_pie_categories=5, n_bar_categories=15, sort_bar_charts_by='count')